# Notebook 04b — DL Tuning & Final Evaluation

**Question:** How much does tuning improve DL models, and how do they compare to ML on the test set?

We tune TabNet with Optuna (the most tunable DL architecture), run 3-fold CV for
reliable DL scores, then evaluate the best DL model on the held-out test set.

**Inputs:** `artifacts/` from nb02, `results/dl_architectures.csv` from nb04a
**Outputs:** `results/dl_tuned.csv`, `results/dl_final_test.csv`, `models/` (best DL model)


In [ ]:
# Uncomment on Colab:
# !pip install pytorch-tabnet -q

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, time, os, joblib
import matplotlib.pyplot as plt, seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

from pytorch_tabnet.tab_model import TabNetClassifier
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

plt.rcParams['figure.figsize'] = (12, 5)
sns.set_style("whitegrid")
SEED = 42; np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs('results', exist_ok=True)
os.makedirs('models', exist_ok=True)
print(f"Device: {DEVICE}")


## 0. Load Data

In [ ]:
X_train_3c = joblib.load('artifacts/X_train_3c.pkl').astype(np.float32)
X_val_3c   = joblib.load('artifacts/X_val_3c.pkl').astype(np.float32)
X_test_3c  = joblib.load('artifacts/X_test_3c.pkl').astype(np.float32)
y_train_3c = joblib.load('artifacts/y_train_3c.pkl')
y_val_3c   = joblib.load('artifacts/y_val_3c.pkl')
y_test_3c  = joblib.load('artifacts/y_test_3c.pkl')

X_train_bin = joblib.load('artifacts/X_train_bin.pkl').astype(np.float32)
X_val_bin   = joblib.load('artifacts/X_val_bin.pkl').astype(np.float32)
X_test_bin  = joblib.load('artifacts/X_test_bin.pkl').astype(np.float32)
y_train_bin = joblib.load('artifacts/y_train_bin.pkl')
y_val_bin   = joblib.load('artifacts/y_val_bin.pkl')
y_test_bin  = joblib.load('artifacts/y_test_bin.pkl')

le_3class     = joblib.load('artifacts/label_encoder_3class.pkl')
feature_names = joblib.load('artifacts/feature_names.pkl')

# For CV: merge train + val
X_3c = np.vstack([X_train_3c, X_val_3c]); y_3c = np.concatenate([y_train_3c, y_val_3c])
X_bin = np.vstack([X_train_bin, X_val_bin]); y_bin = np.concatenate([y_train_bin, y_val_bin])

# Load architecture results to know what to tune
arch_results = pd.read_csv('results/dl_architectures.csv')
print(f'3-class: {X_3c.shape}, Binary: {X_bin.shape}')
print(f'\nArchitecture results from nb04a:')
print(arch_results.sort_values('cv_f1_mean', ascending=False).to_string(index=False))

SCOREBOARD = []
def log_exp(model, strategy, cv_mean, cv_std, n_train, notes=""):
    SCOREBOARD.append(dict(phase="dl_tuning", model=model, strategy=strategy,
        n_features=X_3c.shape[1], cv_f1_mean=round(cv_mean,4),
        cv_f1_std=round(cv_std,4), n_train=n_train, notes=notes))
    print(f"  {model:22s} | {strategy:8s} | F1={cv_mean:.4f} ± {cv_std:.4f} | {notes}")


## 1. Optuna Tuning — TabNet

TabNet has the most impactful hyperparameters among DL architectures for tabular data:
- `n_d` / `n_a`: dimensions of decision and attention layers
- `n_steps`: number of sequential attention steps
- `gamma`: feature reuse coefficient
- `lambda_sparse`: sparsity regularization strength
- `learning_rate`: critical for convergence

We use 3-fold CV inside Optuna for reliable estimates.


In [ ]:
def tabnet_objective_3c(trial):
    n_d_a = trial.suggest_int("n_d_a", 8, 64, step=8)
    params = {
        "n_d": n_d_a,
        "n_a": n_d_a,
        "n_steps": trial.suggest_int("n_steps", 3, 7),
        "gamma": trial.suggest_float("gamma", 1.0, 2.0),
        "lambda_sparse": trial.suggest_float("lambda_sparse", 1e-4, 1e-1, log=True),
        "lr": trial.suggest_float("lr", 5e-3, 5e-2, log=True),
    }

    cv = StratifiedKFold(3, shuffle=True, random_state=SEED)
    scores = []
    for train_idx, val_idx in cv.split(X_3c, y_3c):
        X_tr, X_vl = X_3c[train_idx], X_3c[val_idx]
        y_tr, y_vl = y_3c[train_idx], y_3c[val_idx]

        w = compute_class_weight('balanced', classes=np.unique(y_tr), y=y_tr)

        model = TabNetClassifier(
            n_d=params["n_d"], n_a=params["n_a"],
            n_steps=params["n_steps"], gamma=params["gamma"],
            lambda_sparse=params["lambda_sparse"],
            optimizer_fn=torch.optim.Adam,
            optimizer_params=dict(lr=params["lr"], weight_decay=1e-5),
            scheduler_fn=torch.optim.lr_scheduler.StepLR,
            scheduler_params=dict(step_size=10, gamma=0.9),
            mask_type='sparsemax', verbose=0, seed=SEED,
            device_name=DEVICE.type,
        )
        model.fit(
            X_tr, y_tr, eval_set=[(X_vl, y_vl)],
            eval_metric=['balanced_accuracy'],
            max_epochs=60, patience=10,
            batch_size=256, virtual_batch_size=128,
            weights=dict(enumerate(w)),
        )
        preds = model.predict(X_vl)
        scores.append(f1_score(y_vl, preds, average='macro'))

    return np.mean(scores)

print("Tuning TabNet on 3-class (15 trials, 3-fold CV)...")
print("This may take 10-20 minutes.")
t0 = time.time()
study_3c = optuna.create_study(direction="maximize")
study_3c.optimize(tabnet_objective_3c, n_trials=15)
elapsed = time.time() - t0
print(f"\nBest F1: {study_3c.best_value:.4f} ({elapsed:.0f}s)")
print(f"Best params: {study_3c.best_params}")
log_exp("TabNet-tuned", "3class", study_3c.best_value, 0, len(X_3c), f"15t,3fold,{elapsed:.0f}s")


In [ ]:
# ── Optuna TabNet on binary ──
def tabnet_objective_bin(trial):
    n_d_a = trial.suggest_int("n_d_a", 8, 64, step=8)
    params = {
        "n_d": n_d_a, "n_a": n_d_a,
        "n_steps": trial.suggest_int("n_steps", 3, 7),
        "gamma": trial.suggest_float("gamma", 1.0, 2.0),
        "lambda_sparse": trial.suggest_float("lambda_sparse", 1e-4, 1e-1, log=True),
        "lr": trial.suggest_float("lr", 5e-3, 5e-2, log=True),
    }

    cv = StratifiedKFold(3, shuffle=True, random_state=SEED)
    scores = []
    for train_idx, val_idx in cv.split(X_bin, y_bin):
        X_tr, X_vl = X_bin[train_idx], X_bin[val_idx]
        y_tr, y_vl = y_bin[train_idx], y_bin[val_idx]
        w = compute_class_weight('balanced', classes=np.unique(y_tr), y=y_tr)
        model = TabNetClassifier(
            n_d=params["n_d"], n_a=params["n_a"],
            n_steps=params["n_steps"], gamma=params["gamma"],
            lambda_sparse=params["lambda_sparse"],
            optimizer_fn=torch.optim.Adam,
            optimizer_params=dict(lr=params["lr"], weight_decay=1e-5),
            scheduler_fn=torch.optim.lr_scheduler.StepLR,
            scheduler_params=dict(step_size=10, gamma=0.9),
            mask_type='sparsemax', verbose=0, seed=SEED,
            device_name=DEVICE.type,
        )
        model.fit(X_tr, y_tr, eval_set=[(X_vl, y_vl)],
                  eval_metric=['balanced_accuracy'],
                  max_epochs=60, patience=10, batch_size=256, virtual_batch_size=128,
                  weights=dict(enumerate(w)))
        preds = model.predict(X_vl)
        scores.append(f1_score(y_vl, preds, average='macro'))
    return np.mean(scores)

print("Tuning TabNet on binary (15 trials, 3-fold CV)...")
t0 = time.time()
study_bin = optuna.create_study(direction="maximize")
study_bin.optimize(tabnet_objective_bin, n_trials=15)
print(f"\nBest F1: {study_bin.best_value:.4f} ({time.time()-t0:.0f}s)")
print(f"Best params: {study_bin.best_params}")
log_exp("TabNet-tuned", "binary", study_bin.best_value, 0, len(X_bin), "15t,3fold")

# Save best params
dl_best_params = {
    'tabnet_3c': study_3c.best_params,
    'tabnet_bin': study_bin.best_params,
}
joblib.dump(dl_best_params, 'artifacts/dl_best_params.pkl')


## 2. Final Test Set Evaluation

⚠️ **One shot — no re-tuning after this.**

We train the best DL model (tuned TabNet) on full train+val and evaluate on the test set.


In [ ]:
print("=" * 60)
print("FINAL DL TEST SET EVALUATION")
print("=" * 60)

test_results = []

# ── 3-class: Tuned TabNet ──
print("\n--- 3-class: Tuned TabNet ---")
p = study_3c.best_params
w_3c = compute_class_weight('balanced', classes=np.unique(y_3c), y=y_3c)

final_tabnet_3c = TabNetClassifier(
    n_d=p["n_d_a"], n_a=p["n_d_a"],
    n_steps=p["n_steps"], gamma=p["gamma"],
    lambda_sparse=p["lambda_sparse"],
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=p["lr"], weight_decay=1e-5),
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    scheduler_params=dict(step_size=10, gamma=0.9),
    mask_type='sparsemax', verbose=10, seed=SEED,
    device_name=DEVICE.type,
)
final_tabnet_3c.fit(
    X_3c, y_3c,
    eval_set=[(X_test_3c, y_test_3c)],
    eval_metric=['balanced_accuracy'],
    max_epochs=100, patience=15,
    batch_size=256, virtual_batch_size=128,
    weights=dict(enumerate(w_3c)),
)
y_pred_3c = final_tabnet_3c.predict(X_test_3c)
test_f1_3c = f1_score(y_test_3c, y_pred_3c, average='macro')
print(f"\nTabNet-tuned 3-class Test F1: {test_f1_3c:.4f}")
print(classification_report(y_test_3c, y_pred_3c, target_names=le_3class.classes_))
test_results.append(dict(model="TabNet-tuned", strategy="3class", test_f1=round(test_f1_3c, 4)))

# Confusion matrix
cm = confusion_matrix(y_test_3c, y_pred_3c, normalize='true')
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='.2f', cmap='Purples',
            xticklabels=le_3class.classes_, yticklabels=le_3class.classes_, ax=ax)
ax.set_title('TabNet-tuned — 3-class Test Confusion Matrix')
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.tight_layout()
plt.savefig('results/cm_tabnet_3class_test.png', bbox_inches='tight')
plt.show()

# Save model
final_tabnet_3c.save_model('models/tabnet_tuned_3class')


In [ ]:
# ── Binary: Tuned TabNet ──
print("\n--- Binary: Tuned TabNet ---")
p_bin = study_bin.best_params
w_bin = compute_class_weight('balanced', classes=np.unique(y_bin), y=y_bin)

final_tabnet_bin = TabNetClassifier(
    n_d=p_bin["n_d_a"], n_a=p_bin["n_d_a"],
    n_steps=p_bin["n_steps"], gamma=p_bin["gamma"],
    lambda_sparse=p_bin["lambda_sparse"],
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=p_bin["lr"], weight_decay=1e-5),
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    scheduler_params=dict(step_size=10, gamma=0.9),
    mask_type='sparsemax', verbose=10, seed=SEED,
    device_name=DEVICE.type,
)
final_tabnet_bin.fit(
    X_bin, y_bin,
    eval_set=[(X_test_bin, y_test_bin)],
    eval_metric=['balanced_accuracy'],
    max_epochs=100, patience=15,
    batch_size=256, virtual_batch_size=128,
    weights=dict(enumerate(w_bin)),
)
y_pred_bin = final_tabnet_bin.predict(X_test_bin)
test_f1_bin = f1_score(y_test_bin, y_pred_bin, average='macro')
print(f"\nTabNet-tuned Binary Test F1: {test_f1_bin:.4f}")
print(classification_report(y_test_bin, y_pred_bin, target_names=['failed', 'success']))
test_results.append(dict(model="TabNet-tuned", strategy="binary", test_f1=round(test_f1_bin, 4)))

cm_bin = confusion_matrix(y_test_bin, y_pred_bin, normalize='true')
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm_bin, annot=True, fmt='.2f', cmap='Greens',
            xticklabels=['failed', 'success'], yticklabels=['failed', 'success'], ax=ax)
ax.set_title('TabNet-tuned — Binary Test Confusion Matrix')
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.tight_layout()
plt.savefig('results/cm_tabnet_binary_test.png', bbox_inches='tight')
plt.show()

final_tabnet_bin.save_model('models/tabnet_tuned_binary')


## 3. Save All Results

In [ ]:
# Save scoreboard
sb = pd.DataFrame(SCOREBOARD)
sb.to_csv('results/dl_tuned.csv', index=False)

test_df = pd.DataFrame(test_results)
test_df.to_csv('results/dl_final_test.csv', index=False)

print("Saved files:")
print(f"  results/dl_tuned.csv ({len(sb)} experiments)")
print(f"  results/dl_final_test.csv")
print(f"  models/tabnet_tuned_3class/")
print(f"  models/tabnet_tuned_binary/")
print(f"  artifacts/dl_best_params.pkl")

print("\n" + "=" * 60)
print("DL TUNING SUMMARY")
print("=" * 60)
print(sb.to_string(index=False))
print("\nFinal test scores:")
print(test_df.to_string(index=False))

# Quick ML vs DL preview
print("\n" + "=" * 60)
print("ML vs DL PREVIEW (test scores)")
print("=" * 60)
try:
    ml_test = pd.read_csv('results/ml_final_test.csv')
    all_test = pd.concat([ml_test.assign(type='ML'), test_df.assign(type='DL')])
    print(all_test.to_string(index=False))
    print("\n→ Full comparison in nb05")
except FileNotFoundError:
    print("(ML test results not yet available — run nb03d first)")
